# 02 · Descriptive Statistical Analysis
**Research objectives:** context for Objectives 1–4 (coverage of cities, matrices, platforms and annotations) and Objective 5 (food-related contextual records).

**Rationale:** before any comparison, the analysable evidence base must be described honestly: how many samples, studies, countries and cities are represented, in which matrices, with which sequencing platforms, over which years, and how many carry taxonomic or AMR annotations. These denominators constrain every later claim.

**Input:** `derived/metadata_set.csv` (notebook 01): 8,053 included samples and 840 food-related samples held as a separate contextual dataset.

**Methods:** frequencies, percentages, medians, interquartile ranges (IQR), means and standard deviations. Sequencing depth is summarised on the log10 scale because it spans five orders of magnitude. No inferential tests are performed in this notebook.

In [1]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "scripts" / "amr_common.py").exists():
        sys.path.insert(0, str(_p / "scripts")); break
    if (_p / "Analysis_Results" / "scripts" / "amr_common.py").exists():
        sys.path.insert(0, str(_p / "Analysis_Results" / "scripts")); break
from amr_common import *
NB = "NB02"
nb_header(NB)
import seaborn as sns
meta = pd.read_csv(DER / "metadata_set.csv", low_memory=False)
amr = load_csv("amr_genes_long.csv", usecols=["Sample_ID", "AMR_Reference_Database"])
tax_ids = set(load_csv("taxonomy_long.csv", usecols=["Sample_ID"]).Sample_ID)
div_ids = set(load_csv("diversity_long.csv", usecols=["Sample_ID"]).Sample_ID)
card_ids = set(amr.loc[amr.AMR_Reference_Database == "CARD", "Sample_ID"]); rf_ids = set(amr.loc[amr.AMR_Reference_Database == "ResFinder", "Sample_ID"])
meta["has_coords"] = meta.lat.notna() & meta.lon.notna()
meta["has_CARD_RGI"] = meta.Sample_ID.isin(card_ids); meta["has_ResFinder_KMA"] = meta.Sample_ID.isin(rf_ids)
meta["has_any_AMR"] = meta.has_CARD_RGI | meta.has_ResFinder_KMA
meta["has_taxonomy"] = meta.Sample_ID.isin(tax_ids); meta["has_diversity"] = meta.Sample_ID.isin(div_ids)
meta["city_label"] = np.where(is_missing(meta.City) | is_missing(meta.Country), np.nan, meta.City + " | " + meta.Country)
inc = meta[meta.Inclusion_Status == "Included"].copy(); ctx = meta[meta.Inclusion_Status != "Included"].copy()
print(len(inc), len(ctx))

Notebook NB02 | root = C:\Users\USER\Documents\Research\AMR


{
"pandas": "3.1.0",
"numpy": "2.5.4",
"scipy": "1.18.1",
"statsmodels": "0.15.0",
"sklearn": "1.9.1",
"matplotlib": "3.11.2",
"seaborn": "0.13.2",
"openpyxl": "3.1.5",
"skbio": "0.7.4",
"geopandas": "1.2.0",
"shapely": "2.2.0",
"networkx": "3.7",
"nbformat": "5.11.1",
"nbconvert": "7.17.2",
"nbclient": "0.11.0",
"PIL": "13.0.0"
}


8053 840


## 1. Overall evidence base

In [2]:
def overview(d, label):
    yrs = d.year.dropna()
    return {"Set": label, "Samples": len(d), "Datasets": d.Dataset_ID.nunique(), "Projects (accessions)": d.Project_Accession.nunique(),
            "Countries (reported)": d.loc[~is_missing(d.Country), "Country"].nunique(),
            "Source-labelled cities (City|Country)": d.city_label.nunique(),
            "Continents": d.loc[d.Continent != "Not reported", "Continent"].nunique(),
            "Sample types": d.Sample_Type.nunique(), "Samples with coordinates": int(d.has_coords.sum()),
            "Samples with any AMR gene table": int(d.has_any_AMR.sum()), "  CARD/RGI": int(d.has_CARD_RGI.sum()),
            "  ResFinder/KMA": int(d.has_ResFinder_KMA.sum()), "Samples with taxonomy table": int(d.has_taxonomy.sum()),
            "Samples with diversity indices": int(d.has_diversity.sum()),
            "Collection years (min-max)": f"{int(yrs.min())}-{int(yrs.max())}" if len(yrs) else "Not reported",
            "Samples with collection year": int(d.year.notna().sum())}
ov = pd.DataFrame([overview(inc, "Included urban environmental samples"), overview(ctx, "Food-related contextual samples")]).T
ov.columns = ov.iloc[0]; ov = ov.iloc[1:].reset_index().rename(columns={"index": "Quantity"})
save_table(ov, "D01_evidence_base_overview", "Overview of the analysable evidence base", NB, sheet="Descriptive_Statistics")
ov

[table] Overview of the analysable evidence base: 15 rows -> tables/D01_evidence_base_overview.csv


Set,Quantity,Included urban environmental samples,Food-related contextual samples
0,Samples,8053,840
1,Datasets,112,16
2,Projects (accessions),122,16
3,Countries (reported),106,11
4,Source-labelled cities (City|Country),274,0
5,Continents,7,4
6,Sample types,8,2
7,Samples with coordinates,6134,401
8,Samples with any AMR gene table,1072,155
9,CARD/RGI,447,0


## 2. Distribution by sample type, dataset, country and continent

In [3]:
def depth_stats(s):
    g = s.dropna() / 1e9; lg = np.log10(s.dropna()[s.dropna() > 0])
    return pd.Series({"depth_n": len(g), "median_Gbp": g.median(), "Q1_Gbp": g.quantile(.25), "Q3_Gbp": g.quantile(.75),
                      "mean_log10_bases": lg.mean(), "sd_log10_bases": lg.std()})
bytype = inc.groupby("Sample_Type").agg(samples=("Sample_ID", "size"), datasets=("Dataset_ID", "nunique"),
        countries=("Country", lambda s: s[~is_missing(s)].nunique()), cities=("city_label", "nunique"),
        with_coordinates=("has_coords", "sum"), with_AMR=("has_any_AMR", "sum"), with_taxonomy=("has_taxonomy", "sum"),
        year_min=("year", "min"), year_max=("year", "max")).join(inc.groupby("Sample_Type").bases.apply(depth_stats).unstack())
bytype.insert(1, "pct_of_included", (100 * bytype.samples / len(inc)).round(1))
bytype = bytype.sort_values("samples", ascending=False).reset_index()
save_table(bytype.pipe(rnd, 3), "D02_by_sample_type", "Included samples by sample type: coverage and sequencing depth", NB, sheet="Sample_Distribution")
bytype

[table] Included samples by sample type: coverage and sequencing depth: 8 rows -> tables/D02_by_sample_type.csv


,Sample_Type,samples,pct_of_included,datasets,countries,cities,with_coordinates,with_AMR,with_taxonomy,year_min,year_max,depth_n,median_Gbp,Q1_Gbp,Q3_Gbp,mean_log10_bases,sd_log10_bases
0,Municipal wastewater,2254,28.0,35,104,243,1908,522,0,2012.0,2024.0,2254.0,10.509812,3.411036,13.923191,9.710048,0.707469
1,Transit surface,1911,23.7,4,14,32,1180,24,0,2013.0,2022.0,1911.0,1.233858,0.819752,1.850681,9.092177,0.487565
2,Built environment,1449,18.0,9,10,0,1374,9,0,2011.0,2024.0,1449.0,1.292474,0.568734,3.872796,9.029899,0.729572
3,Surface water,882,11.0,36,15,0,830,46,0,2012.0,2025.0,882.0,2.587247,0.037615,14.858134,8.845222,1.598503
4,Community wastewater,529,6.6,6,4,5,82,447,447,2014.0,2025.0,529.0,5.451807,4.386949,6.679848,9.742058,0.151209
5,Air,427,5.3,7,10,0,392,24,0,2012.0,2025.0,427.0,1.556664,1.226292,4.179971,9.315858,0.353146
6,Urban soil,364,4.5,7,7,0,291,0,0,2013.0,2024.0,364.0,10.710765,3.070582,16.315578,9.814224,0.551977
7,Hospital wastewater,237,2.9,16,11,1,77,0,0,2018.0,2026.0,237.0,0.220991,0.051253,4.029354,8.314778,1.535543


In [4]:
names = meta.drop_duplicates("Dataset_ID").set_index("Dataset_ID").Dataset_Name
byds = meta.groupby(["Dataset_ID", "Inclusion_Status"]).agg(samples=("Sample_ID", "size"),
        sample_types=("Sample_Type", lambda s: "; ".join(sorted(s.unique()))), countries=("Country", lambda s: s[~is_missing(s)].nunique()),
        cities=("city_label", "nunique"), platform=("Platform_Family", lambda s: "; ".join(sorted(s.unique()))),
        year_min=("year", "min"), year_max=("year", "max"), CARD_RGI=("has_CARD_RGI", "sum"), ResFinder_KMA=("has_ResFinder_KMA", "sum"),
        taxonomy=("has_taxonomy", "sum")).reset_index()
byds.insert(1, "Dataset_Name", byds.Dataset_ID.map(names))
byds = byds.sort_values("samples", ascending=False)
save_table(byds, "D03_by_dataset", "Samples by dataset (included and contextual)", NB, sheet="Sample_Distribution")
byds.head(15)

[table] Samples by dataset (included and contextual): 128 rows -> tables/D03_by_dataset.csv


,Dataset_ID,Dataset_Name,Inclusion_Status,samples,sample_types,countries,cities,platform,year_min,year_max,CARD_RGI,ResFinder_KMA,taxonomy
3,DS04,MetaSUB City Sampling Day 2016-2017 (SRA subset),Included,1804,Transit surface,12,32,Illumina,2016.0,2017.0,0,0,0
0,DS01,Global Sewage Surveillance Project (Munk et al...,Included,1034,Municipal wastewater,102,243,Illumina,2016.0,2019.0,0,315,0
1,DS02,Indian metropolitan open-drain wastewater (Sin...,Included,447,Community wastewater,1,4,Illumina,2022.0,2024.0,447,0,447
26,DS115,Hong Kong outdoor built environment surface me...,Included,415,Built environment,1,0,Illumina,2017.0,2017.0,0,0,0
22,DS111,Shotgun metagenomic sequencing of subway and u...,Included,294,Air,5,0,Illumina,2017.0,2017.0,0,0,0
106,DS78,Examining the microbiome and resistome of an u...,Included,279,Surface water,1,0,Oxford Nanopore,2012.0,2024.0,0,0,0
90,DS62,Metagenomic Insights into Microbial Communitie...,Included,272,Built environment,1,0,Illumina,2021.0,2023.0,0,0,0
4,DS05,FDA retail lamb food metagenome (contextual),Separate contextual dataset,263,Food-related environment,1,0,Illumina,2024.0,2025.0,0,0,0
5,DS06,Urban microbiomes in Japan,Included,261,Built environment,1,0,DNBSEQ/MGI/BGISEQ,2021.0,2022.0,0,0,0
114,DS86,Metagenomic sequencing of urban wastewater sam...,Included,252,Municipal wastewater,1,0,DNBSEQ/MGI/BGISEQ,2023.0,2023.0,0,0,0


In [5]:
bycountry = inc[~is_missing(inc.Country)].groupby(["Continent", "Country"]).agg(samples=("Sample_ID", "size"),
        sample_types=("Sample_Type", "nunique"), cities=("city_label", "nunique"), datasets=("Dataset_ID", "nunique"),
        with_AMR=("has_any_AMR", "sum")).reset_index().sort_values("samples", ascending=False)
save_table(bycountry, "D04_by_country", "Included samples by country", NB, sheet="Geographic_Coverage")
ct = pd.crosstab(inc.Continent, inc.Sample_Type, margins=True, margins_name="Total")
save_table(ct.reset_index(), "D05_continent_by_sample_type", "Included samples: continent by sample type", NB, sheet="Geographic_Coverage")
top_c = bycountry.head(10); print(f"Top 10 countries hold {top_c.samples.sum()} of {len(inc)} included samples ({100*top_c.samples.sum()/len(inc):.1f}%)")
ct

[table] Included samples by country: 106 rows -> tables/D04_by_country.csv
[table] Included samples: continent by sample type: 9 rows -> tables/D05_continent_by_sample_type.csv
Top 10 countries hold 5611 of 8053 included samples (69.7%)


Sample_Type,Air,Built environment,Community wastewater,Hospital wastewater,Municipal wastewater,Surface water,Transit surface,Urban soil,Total
Continent,,,,,,,,,
Africa,0,271,46,20,445,0,43,20,845
Asia,164,1042,453,172,588,270,209,190,3088
Europe,192,132,30,21,867,36,654,134,2066
North America,59,1,0,0,165,529,655,20,1429
Not reported,12,3,0,16,15,15,208,0,269
Oceania,0,0,0,0,83,14,33,0,130
Seven seas (open ocean),0,0,0,0,6,0,0,0,6
South America,0,0,0,8,85,18,109,0,220
Total,427,1449,529,237,2254,882,1911,364,8053


In [6]:
bycity = inc[inc.city_label.notna()].groupby("city_label").agg(samples=("Sample_ID", "size"), sample_types=("Sample_Type", lambda s: "; ".join(sorted(s.unique()))),
        datasets=("Dataset_ID", lambda s: "; ".join(sorted(s.unique()))), with_AMR=("has_any_AMR", "sum"), with_taxonomy=("has_taxonomy", "sum")).reset_index().sort_values("samples", ascending=False)
save_table(bycity, "D06_by_city", "Included samples by source-labelled city", NB, sheet="Geographic_Coverage")
city_n = bycity.samples
city_sum = pd.DataFrame([{"Quantity": "Source-labelled cities", "Value": len(bycity)},
    {"Quantity": "Cities with >=10 samples", "Value": int((city_n >= 10).sum())},
    {"Quantity": "Cities with >=30 samples", "Value": int((city_n >= 30).sum())},
    {"Quantity": "Cities with 1 sample", "Value": int((city_n == 1).sum())},
    {"Quantity": "Median samples per city (IQR)", "Value": f"{city_n.median():.0f} ({city_n.quantile(.25):.0f}-{city_n.quantile(.75):.0f})"},
    {"Quantity": "Cities represented in >1 sample type", "Value": int((bycity.sample_types.str.count(";") >= 1).sum())},
    {"Quantity": "Cities with >=1 AMR-annotated sample", "Value": int((bycity.with_AMR > 0).sum())},
    {"Quantity": "Cities with >=1 taxonomy-annotated sample", "Value": int((bycity.with_taxonomy > 0).sum())},
    {"Quantity": "Included samples without a City|Country pair (city missing, or country missing/unverified)", "Value": int(inc.city_label.isna().sum())}])
save_table(city_sum, "D07_city_coverage_summary", "City coverage summary (relevance to the proposal's 35-city framing)", NB, sheet="Geographic_Coverage")
city_sum

[table] Included samples by source-labelled city: 274 rows -> tables/D06_by_city.csv
[table] City coverage summary (relevance to the proposal's 35-city framing): 9 rows -> tables/D07_city_coverage_summary.csv


,Quantity,Value
0,Source-labelled cities,274
1,Cities with >=10 samples,33
2,Cities with >=30 samples,12
3,Cities with 1 sample,67
4,Median samples per city (IQR),3 (2-5)
5,Cities represented in >1 sample type,6
6,Cities with >=1 AMR-annotated sample,79
7,Cities with >=1 taxonomy-annotated sample,4
8,Included samples without a City|Country pair (...,5069


## 3. Proposal target cities (Dhaka, Delhi, Karachi)

In [7]:
loc = load_csv("sample_master.csv", usecols=["Sample_ID", "Locality_As_Reported"]).set_index("Sample_ID").Locality_As_Reported
meta["locality"] = meta.Sample_ID.map(loc)
tg = []
for c in ["Dhaka", "Delhi", "Karachi"]:
    m = meta[(meta.City.astype(str).str.contains(c, case=False)) | (meta.locality.astype(str).str.contains(c, case=False))]
    for (ds, st, inc_s), g in m.groupby(["Dataset_ID", "Sample_Type", "Inclusion_Status"]):
        tg.append(dict(target_city=c, Dataset_ID=ds, Dataset_Name=names.get(ds), Sample_Type=st, Inclusion_Status=inc_s, samples=len(g),
                       source_labelled_city=int(g.City.astype(str).str.contains(c, case=False).sum()),
                       years=f"{g.year.min():.0f}-{g.year.max():.0f}" if g.year.notna().any() else "Not reported",
                       CARD_RGI=int(g.has_CARD_RGI.sum()), ResFinder_KMA=int(g.has_ResFinder_KMA.sum()), taxonomy=int(g.has_taxonomy.sum())))
tg = pd.DataFrame(tg)
save_table(tg, "D08_target_cities", "Records for the proposal's named cities (Dhaka, Delhi, Karachi)", NB, sheet="Geographic_Coverage")
tg

[table] Records for the proposal's named cities (Dhaka, Delhi, Karachi): 12 rows -> tables/D08_target_cities.csv


,target_city,Dataset_ID,Dataset_Name,Sample_Type,Inclusion_Status,samples,source_labelled_city,years,CARD_RGI,ResFinder_KMA,taxonomy
0,Dhaka,DS01,Global Sewage Surveillance Project (Munk et al...,Municipal wastewater,Included,4,4,2017-2018,0,0,0
1,Dhaka,DS03,Dhaka hospital and community wastewater (BCSIR),Community wastewater,Included,1,1,2025-2025,0,0,0
2,Dhaka,DS03,Dhaka hospital and community wastewater (BCSIR),Hospital wastewater,Included,3,3,2025-2025,0,0,0
3,Dhaka,DS84,Shotgun Metagenomics of Hospital and Urban Was...,Hospital wastewater,Included,4,0,2024-2024,0,0,0
4,Dhaka,DS84,Shotgun Metagenomics of Hospital and Urban Was...,Municipal wastewater,Included,3,0,2024-2024,0,0,0
5,Dhaka,DS84,Shotgun Metagenomics of Hospital and Urban Was...,Surface water,Included,1,0,2024-2024,0,0,0
6,Delhi,DS02,Indian metropolitan open-drain wastewater (Sin...,Community wastewater,Included,100,100,2022-2024,100,0,100
7,Delhi,DS107,Sediment Metagenome of River Yamuna (Wazaribad...,Surface water,Included,1,0,2017-2017,0,0,0
8,Delhi,DS108,Sediment Metagenome of River Yamuna (Faizupur ...,Surface water,Included,1,0,2017-2017,0,0,0
9,Delhi,DS43,New Delhi House Microbiome,Built environment,Included,90,0,2021-2021,0,0,0


## 4. Sequencing platforms, read types and temporal coverage

In [8]:
plat = pd.crosstab(inc.Platform_Family, inc.Sample_Type, margins=True, margins_name="Total")
save_table(plat.reset_index(), "D09_platform_by_sample_type", "Platform family by sample type (included samples)", NB, sheet="Descriptive_Statistics")
rt = inc.Read_Type.fillna("Not reported").value_counts().rename_axis("Read_Type").reset_index(name="samples")
save_table(rt, "D10_read_type", "Read layout (included samples)", NB, sheet="Descriptive_Statistics")
yr = pd.crosstab(inc.year.fillna(-1).astype(int).replace(-1, "Not reported").astype(str), inc.Sample_Type)
save_table(yr.reset_index().rename(columns={"year": "Collection_Year"}), "D11_year_by_sample_type", "Collection year by sample type (included samples)", NB, sheet="Descriptive_Statistics")
plat

[table] Platform family by sample type (included samples): 6 rows -> tables/D09_platform_by_sample_type.csv


[table] Read layout (included samples): 3 rows -> tables/D10_read_type.csv
[table] Collection year by sample type (included samples): 17 rows -> tables/D11_year_by_sample_type.csv


Sample_Type,Air,Built environment,Community wastewater,Hospital wastewater,Municipal wastewater,Surface water,Transit surface,Urban soil,Total
Platform_Family,,,,,,,,,
DNBSEQ/MGI/BGISEQ,0,261,0,17,297,25,0,90,690
Illumina,427,1188,529,188,1939,546,1911,274,7002
Ion Torrent,0,0,0,0,12,0,0,0,12
Oxford Nanopore,0,0,0,32,0,311,0,0,343
PacBio,0,0,0,0,6,0,0,0,6
Total,427,1449,529,237,2254,882,1911,364,8053


In [9]:
order = bytype.Sample_Type.tolist()
fig, axes = plt.subplots(1, 2, figsize=(6.9, 3.4), gridspec_kw={"width_ratios": [1.25, 1]})
cont_order = inc.Continent.value_counts().index.tolist()
cpal = dict(zip(cont_order, sns.color_palette("Set2", len(cont_order))))
tab = pd.crosstab(inc.Sample_Type, inc.Continent).reindex(order)[cont_order]
left = np.zeros(len(tab))
for c in cont_order:
    axes[0].barh(tab.index, tab[c], left=left, color=cpal[c], label=c, edgecolor="white", linewidth=.3); left += tab[c].values
for i, v in enumerate(left): axes[0].text(v + 15, i, f"{int(v):,}", va="center", fontsize=6.8)
axes[0].invert_yaxis(); axes[0].set_xlabel("Included samples (n)"); axes[0].set_title("A  Sample type by continent")
axes[0].legend(frameon=False, fontsize=6.5, loc="lower right")
ptab = pd.crosstab(inc.Sample_Type, inc.Platform_Family).reindex(order)
ppct = ptab.div(ptab.sum(axis=1), axis=0) * 100
pcols = ppct.columns.tolist(); ppal = dict(zip(pcols, sns.color_palette("Dark2", len(pcols))))
left = np.zeros(len(ppct))
for c in pcols:
    axes[1].barh(ppct.index, ppct[c], left=left, color=ppal[c], label=c, edgecolor="white", linewidth=.3); left += ppct[c].values
axes[1].invert_yaxis(); axes[1].set_yticklabels([]); axes[1].set_xlabel("% of samples"); axes[1].set_title("B  Sequencing platform family")
axes[1].legend(frameon=False, fontsize=6.5, loc="upper center", bbox_to_anchor=(.5, -.15), ncol=3)
fig.tight_layout()
save_fig(fig, 2, "sample_type_continent_platform", "Included samples by sample type, continent and sequencing platform",
         f"(A) Number of included samples (n = {len(inc):,}) per sample type, stacked by continent (Natural Earth continent of the reported country; 'Not reported' where country is missing). (B) Percentage of each sample type sequenced on each platform family.",
         NB, "Objective 1 (context)", "tables/D02_by_sample_type.csv; tables/D09_platform_by_sample_type.csv")

[figure] Figure 2: Included samples by sample type, continent and sequencing platform -> figures/Fig02_sample_type_continent_platform.png (177 x 85 mm; min font 6.5 pt; overlaps 0)


'Fig02_sample_type_continent_platform.png'

In [10]:
fig, ax = plt.subplots(figsize=(6.9, 3.0))
y = inc[inc.year.notna()]; years = np.arange(int(y.year.min()), int(y.year.max()) + 1)
bottom = np.zeros(len(years))
for st in order:
    v = y[y.Sample_Type == st].year.value_counts().reindex(years, fill_value=0).values
    ax.bar(years, v, bottom=bottom, color=SAMPLE_TYPE_COLORS.get(st), label=st, width=.8, edgecolor="white", linewidth=.3); bottom += v
ax.set_xlabel("Collection year"); ax.set_ylabel("Included samples (n)"); ax.set_xticks(years[::2]); ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False, ncol=2, fontsize=6.5); ax.set_title("Temporal coverage of included samples")
save_fig(fig, 3, "temporal_coverage", "Collection year of included samples by sample type",
         f"Number of included samples per reported collection year, stacked by sample type (n = {len(y):,} of {len(inc):,} with a collection year; placeholder dates excluded).",
         NB, "Objective 1 / 3 (context)", "tables/D11_year_by_sample_type.csv")

[figure] Figure 3: Collection year of included samples by sample type -> figures/Fig03_temporal_coverage.png (152 x 79 mm; min font 6.5 pt; overlaps 0)


'Fig03_temporal_coverage.png'

## 5. Annotation coverage and sequencing depth

In [11]:
cov = inc.groupby("Sample_Type")[["has_coords", "has_any_AMR", "has_CARD_RGI", "has_ResFinder_KMA", "has_taxonomy", "has_diversity"]].sum()
cov.insert(0, "samples", inc.groupby("Sample_Type").size()); cov = cov.reindex(order)
cov.loc["Total"] = cov.sum()
covp = cov.copy()
for c in cov.columns[1:]: covp[c + "_pct"] = (100 * cov[c] / cov.samples).round(1)
save_table(covp.reset_index(), "D12_annotation_coverage_by_type", "Coordinates, AMR, taxonomy and diversity coverage by sample type (included samples)", NB, sheet="Descriptive_Statistics")
fig, axes = plt.subplots(1, 2, figsize=(6.9, 3.2))
c2 = cov.drop(index="Total")
w = .16; x = np.arange(len(c2))
for i, (col, lab) in enumerate([("samples", "All included"), ("has_coords", "Coordinates"), ("has_any_AMR", "AMR gene table"), ("has_taxonomy", "Taxonomy table"), ("has_diversity", "Diversity indices")]):
    axes[0].bar(x + (i - 2) * w, c2[col].replace(0, np.nan), w, label=lab, color=sns.color_palette("Blues_r", 6)[i] if i else "#333333")
axes[0].set_yscale("log"); axes[0].set_xticks(x); axes[0].set_xticklabels(c2.index, rotation=35, ha="right"); axes[0].set_ylabel("Samples (log scale)")
axes[0].legend(frameon=False, fontsize=6.5); axes[0].set_title("A  Annotation coverage")
dd = inc[inc.bases > 0].assign(Gbp=lambda d: d.bases / 1e9)
sns.boxplot(data=dd, y="Sample_Type", x="Gbp", order=order, hue="Sample_Type", palette=SAMPLE_TYPE_COLORS, legend=False, fliersize=1.2, linewidth=.7, ax=axes[1], log_scale=True)
axes[1].set_xlabel("Sequenced bases per sample (Gbp, log scale)"); axes[1].set_ylabel(""); axes[1].set_title("B  Sequencing depth")
fig.tight_layout()
save_fig(fig, 4, "annotation_coverage_depth", "Annotation coverage and sequencing depth by sample type",
         "(A) Included samples per sample type with source-reported coordinates, a gene-level AMR table (CARD/RGI or ResFinder/KMA), a taxonomic table, or diversity indices (log scale; absent bars = zero). (B) Total sequenced bases per sample (Gbp, log scale); boxes show median and IQR. Most samples have metadata and raw reads only; processed AMR or taxonomic tables exist for a minority.",
         NB, "Objectives 1-2 (data availability)", "tables/D12_annotation_coverage_by_type.csv; tables/D02_by_sample_type.csv")
covp

[table] Coordinates, AMR, taxonomy and diversity coverage by sample type (included samples): 9 rows -> tables/D12_annotation_coverage_by_type.csv


[figure] Figure 4: Annotation coverage and sequencing depth by sample type -> figures/Fig04_annotation_coverage_depth.png (174 x 81 mm; min font 6.5 pt; overlaps 1)


,samples,has_coords,has_any_AMR,has_CARD_RGI,has_ResFinder_KMA,has_taxonomy,has_diversity,has_coords_pct,has_any_AMR_pct,has_CARD_RGI_pct,has_ResFinder_KMA_pct,has_taxonomy_pct,has_diversity_pct
Sample_Type,,,,,,,,,,,,,
Municipal wastewater,2254,1908,522,0,522,0,522,84.6,23.2,0.0,23.2,0.0,23.2
Transit surface,1911,1180,24,0,24,0,24,61.7,1.3,0.0,1.3,0.0,1.3
Built environment,1449,1374,9,0,9,0,9,94.8,0.6,0.0,0.6,0.0,0.6
Surface water,882,830,46,0,46,0,46,94.1,5.2,0.0,5.2,0.0,5.2
Community wastewater,529,82,447,447,0,447,0,15.5,84.5,84.5,0.0,84.5,0.0
Air,427,392,24,0,24,0,24,91.8,5.6,0.0,5.6,0.0,5.6
Urban soil,364,291,0,0,0,0,0,79.9,0.0,0.0,0.0,0.0,0.0
Hospital wastewater,237,77,0,0,0,0,0,32.5,0.0,0.0,0.0,0.0,0.0
Total,8053,6134,1072,447,625,447,625,76.2,13.3,5.6,7.8,5.6,7.8


In [12]:
n_amr = int(inc.has_any_AMR.sum()); n_tax = int(inc.has_taxonomy.sum())
record_finding(NB, "F01", "Objective 1-4 (evidence base)",
    f"The analysable evidence base comprises {len(inc):,} included urban environmental shotgun metagenomes from {inc.Dataset_ID.nunique()} datasets, {inc.loc[~is_missing(inc.Country),'Country'].nunique()} countries and {inc.city_label.nunique()} source-labelled cities, but only {n_amr:,} ({100*n_amr/len(inc):.1f}%) carry a processed AMR gene table and {n_tax} ({100*n_tax/len(inc):.1f}%) a taxonomic table.",
    len(inc), "Included samples", "Descriptive counts", "Table D01, D12; Figure 4",
    "Most records support only metadata-level description; taxonomic and AMR inference rests on two processed sub-collections.",
    "Raw reads (approx. 24 TB) were not downloaded or reprocessed; processed tables come from two non-equivalent pipelines.", "Descriptive")
record_finding(NB, "F02", "Objective 1 (geographic coverage)",
    f"Coverage is geographically uneven: the ten best-sampled countries contribute {top_c.samples.sum():,} of {len(inc):,} included samples ({100*top_c.samples.sum()/len(inc):.1f}%); {int((city_n == 1).sum())} of {len(bycity)} source-labelled cities contribute a single sample.",
    len(inc), "Included samples with a reported country / city", "Descriptive counts", "Tables D04, D06, D07",
    "City-level comparisons and rankings are not supported for most cities.", "City labels are source-reported; 625 locality strings were not standardized.", "Descriptive")
tgs = tg.groupby("target_city")[["samples", "CARD_RGI", "ResFinder_KMA", "taxonomy"]].sum()
desc = "; ".join(f"{c}: {int(r.samples)} samples ({int(r.CARD_RGI)} CARD/RGI, {int(r.ResFinder_KMA)} ResFinder/KMA, {int(r.taxonomy)} taxonomy)" for c, r in tgs.iterrows())
record_finding(NB, "F03", "Objective 1 (proposal target cities)",
    f"All three named cities are present, but processed annotation is sparse: {desc}.",
    int(tg.samples.sum()), "Included and contextual samples labelled with a target city (City or locality text)", "Descriptive counts", "Table D08",
    "Target-city AMR and taxonomy statements must be restricted to the cities with processed tables; the rest require raw-read reprocessing.",
    "Small and heterogeneous samples; locality strings are not standardized.", "Descriptive")
print(tgs)

             samples  CARD_RGI  ResFinder_KMA  taxonomy
target_city                                            
Delhi            202       100              0       100
Dhaka             16         0              0         0
Karachi            8         0              1         0
